**# 04 — Incremental Load
 Fetches yesterday's weather data from Open-Meteo API
For all 5 cities
Appends to Bronze → Merges into Silver
Runs daily — fully idempotent**

In [0]:
# 04_incremental

from datetime import datetime, timedelta
import requests
import time

dbutils.widgets.text("target_date",
    (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d"),
    "Target Date (YYYY-MM-DD)")

TARGET_DATE = dbutils.widgets.get("target_date")
BATCH_ID    = f"batch_incremental_{TARGET_DATE}"
RAW_DIR     = "dbfs:/Volumes/workspace/default/products/incremental"

print(f"Target Date : {TARGET_DATE}")
print(f"Batch ID    : {BATCH_ID}")

In [0]:
CITIES = [
    {"name": "Lahore",    "lat": 31.5204, "lon": 74.3587},
    {"name": "Karachi",   "lat": 24.8607, "lon": 67.0011},
    {"name": "Islamabad", "lat": 33.6844, "lon": 73.0479},
    {"name": "Peshawar",  "lat": 34.0151, "lon": 71.5249},
    {"name": "Quetta",    "lat": 30.1798, "lon": 66.9750},
]

VARIABLES = ",".join([
    "temperature_2m", "relative_humidity_2m", "dew_point_2m", "apparent_temperature",
    "pressure_msl", "surface_pressure", "weather_code", "vapour_pressure_deficit",
    "precipitation", "rain", "snowfall", "snow_depth",
    "wind_speed_10m", "wind_speed_100m", "wind_direction_10m", "wind_direction_100m", "wind_gusts_10m",
    "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high",
    "shortwave_radiation", "direct_radiation", "diffuse_radiation",
    "direct_normal_irradiance", "global_tilted_irradiance", "terrestrial_radiation",
    "shortwave_radiation_instant", "direct_radiation_instant", "diffuse_radiation_instant",
    "direct_normal_irradiance_instant", "global_tilted_irradiance_instant", "terrestrial_radiation_instant",
    "soil_temperature_0_to_7cm", "soil_temperature_7_to_28cm",
    "soil_temperature_28_to_100cm", "soil_temperature_100_to_255cm",
    "soil_moisture_0_to_7cm", "soil_moisture_7_to_28cm",
    "soil_moisture_28_to_100cm", "soil_moisture_100_to_255cm",
    "et0_fao_evapotranspiration"
])

raw_payloads = []
print(f"Fetching {TARGET_DATE} for all cities...")

for city in CITIES:
    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={city['lat']}&longitude={city['lon']}"
        f"&start_date={TARGET_DATE}&end_date={TARGET_DATE}"
        f"&hourly={VARIABLES}"
        f"&timezone=Asia/Karachi"
    )
    print(f"  {city['name']}...", end=" ")
    for attempt in range(1, 6):
        try:
            r = requests.get(url, timeout=60)
            if r.status_code == 200:
                raw_payloads.append(r.text)
                print("✓")
                break
            elif r.status_code == 429:
                wait = 60 * attempt
                print(f"Rate limited. Waiting {wait}s...")
                time.sleep(wait)
        except Exception as e:
            print(f"Error: {e}. Waiting 30s...")
            time.sleep(30)
    time.sleep(5)

print(f"Fetched {len(raw_payloads)}/5 cities!")

In [0]:
dbutils.fs.mkdirs(RAW_DIR)

RAW_FILE = f"{RAW_DIR}/incremental_{TARGET_DATE}.json"
combined = "[\n" + ",\n".join(raw_payloads) + "\n]"
temp     = f"{RAW_DIR}/temp"

spark.createDataFrame([(combined,)], ["value"]) \
    .coalesce(1).write.mode("overwrite").text(temp)

for f in dbutils.fs.ls(temp):
    if f.name.startswith("part-"):
        dbutils.fs.mv(f.path, RAW_FILE)
    elif f.name.startswith("_"):
        dbutils.fs.rm(f.path)

dbutils.fs.rm(temp, True)

size = dbutils.fs.ls(RAW_FILE)[0].size / 1024 / 1024
print(f"Raw file saved: {RAW_FILE} ({size:.2f} MB)")

In [0]:
# Set Python variables for Bronze (shared via %run namespace)
# No widgets — serverless widget values are stale and can't be updated reliably
LOAD_TYPE = "incremental"
BATCH_ID  = BATCH_ID   # already set in Cell 2
RAW_FILE  = RAW_FILE   # already set in Cell 4

print("Bronze vars set: incremental")
print(f"  batch_id : {BATCH_ID}")
print(f"  raw_file : {RAW_FILE}")

In [0]:
%run /Workspace/Users/ibrahimqaiser392@gmail.com/Pakistan-5-citites-Weather-Analytics-Lakehouse-Pipeline/phase2/notebooks/02-bronzelayer

In [0]:
# Set Python variables for Silver (shared via %run namespace)
LOAD_TYPE = "incremental"
BATCH_ID  = f"batch_incremental_{TARGET_DATE}"

print("Silver vars set: incremental")
print(f"  batch_id : {BATCH_ID}")

In [0]:
%run /Workspace/Users/ibrahimqaiser392@gmail.com/Pakistan-5-citites-Weather-Analytics-Lakehouse-Pipeline/phase2/notebooks/03-silverlayer

In [0]:
print("Silver complete!")
print(f"\n{'='*40}")
print(f"INCREMENTAL DONE! Date: {TARGET_DATE}")
print(f"{'='*40}")

In [0]:
print("\nAll pipeline logs:")
spark.sql("""
    SELECT *
    FROM pakistan_weather.pipeline_execution_logs
    ORDER BY log_id
""").show(truncate=False)

# Check row counts for both Bronze and Silver tables
bronze_count = spark.table("pakistan_weather.bronze_weather_raw").count()
silver_count = spark.table("pakistan_weather.silver_weather_cleaned").count()

print(f"Bronze layer count: {bronze_count:,} rows")
print(f"Silver layer count: {silver_count:,} rows")